# Deduplication — Removing Duplicate Rows

Remove duplicate records across the intermediate DataFrame with full control over column subsets.

In pandas, deduplication is performed via `df.drop_duplicates()`, and integrates naturally into `pytae` method chains. In the CLI, the equivalent flag is `-dedupe` (or `-dedupe COLS`).

| Operation | Pandas / Method Chaining | CLI Equivalent |
|---|---|---|
| All Columns | `df.drop_duplicates().reset_index(drop=True)` | `-dedupe` |
| Specific Columns | `df.drop_duplicates(subset=['col1', 'col2']).reset_index(drop=True)` | `-dedupe "col1,col2"` |
| Keep Last / None | `df.drop_duplicates(keep='last')` | N/A |

---

In [ ]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt
import pandas as pd

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']

## 1. Deduplicating Across All Columns

Eliminate identical rows across all selected columns:

In [ ]:
# Keep only unique species and island pairs
(
    penguins
    .pt.select("species", "island")
    .drop_duplicates()
    .reset_index(drop=True)
)

## 2. Deduplicating by Specific Identifier Columns (`subset=`)

Restrict the uniqueness constraint to specific columns, keeping the first row for each group:

In [ ]:
# Keep the first recorded penguin for each unique species and island combination
(
    penguins
    .pt.select("species", "island", "body_mass_g", "sex")
    .drop_duplicates(subset=["species", "island"])
    .reset_index(drop=True)
)

## 3. Chaining in Pipelines

Deduplication works cleanly between queries, transforms, and sorting:

In [ ]:
# Filter heavy penguins -> select columns -> deduplicate -> arrange
(
    penguins
    .pt.qry("body_mass_g > 4000")
    .pt.select("species", "island")
    .drop_duplicates()
    .pt.arrange("species", "island")
    .reset_index(drop=True)
)

## 4. Keeping the Last Record (`keep='last'`)

Pandas allows keeping the last occurrence instead of the first:

In [ ]:
# Retain the last record for each day and time
(
    tips
    .pt.select("day", "time", "total_bill", "tip")
    .drop_duplicates(subset=["day", "time"], keep="last")
    .reset_index(drop=True)
)